# Procesamiento de Imágenes mediante la Transformada Rápida de Fourier (FFT 2D)

**Universidad Centroamericana "José Simeón Cañas"**  
Facultad de Ingeniería y Arquitectura — Departamento de Matemática  
Análisis Numérico | Ciclo 01-2026  
Sección 01 | Ing. Daniel Augusto Sosa

---

### Integrantes
- Maria Auxiliadora Chinchilla Flores
- Darlyn Maricela Donis Palma
- Diana Alejandra Erazo Pérez
- Luis Eduardo Martinez Linares
- Rebeca Elizabeth Torres Angel

---

## Descripción del proyecto

Este notebook demuestra la aplicación de la **Transformada Rápida de Fourier en 2 dimensiones (FFT 2D)** al procesamiento digital de imágenes. El objetivo es analizar una imagen en el **dominio de frecuencias**, diseñar filtros espectrales y reconstruir la imagen procesada mediante la **Transformada Inversa (IFFT 2D)**.

El flujo de trabajo sigue este ciclo:

```
Imagen original
      ↓
  FFT 2D          →  dominio de frecuencias
      ↓
  fftshift        →  centrar el espectro (DC en el centro)
      ↓
  Máscara/Filtro  →  pasa bajas, pasa altas, o notch
      ↓
  ifftshift + IFFT →  regresar al dominio espacial
      ↓
  Imagen procesada
```

### Librerías utilizadas

| Librería | Propósito |
|---|---|
| `numpy` | Representación matricial de imágenes y cálculo de FFT 2D |
| `matplotlib` | Visualización de imágenes y espectros |
| `scipy.fftpack` | Alternativa optimizada para FFT 2D |
| `PIL (Pillow)` | Carga y conversión de archivos de imagen |
| `skimage` | Imágenes de prueba estándar y métricas de calidad |

---
## Sección 0 — Importación de librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import data, metrics
import warnings
warnings.filterwarnings('ignore')

# Configuración global de matplotlib para gráficas más claras
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 12

print("✓ Librerías importadas correctamente")
print(f"  NumPy  versión: {np.__version__}")

---
## Sección 1 — Fundamento Matemático

### 1.1 La DFT en una dimensión

Para una señal discreta $x[n]$ de longitud $N$, la **Transformada Discreta de Fourier (DFT)** se define como:

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j\,2\pi kn/N}, \qquad k = 0, 1, \ldots, N-1$$

donde $j = \sqrt{-1}$ es la unidad imaginaria. Cada coeficiente $X[k]$ representa la contribución de la frecuencia $k/N$ a la señal original.

### 1.2 Extensión a 2 dimensiones — DFT 2D

Una imagen en escala de grises es una matriz $x[n_1, n_2]$ de tamaño $M \times N$. La **DFT 2D** se define como:

$$X[k_1, k_2] = \sum_{n_1=0}^{M-1} \sum_{n_2=0}^{N-1} x[n_1, n_2]\, e^{-j2\pi\!\left(\frac{k_1 n_1}{M} + \frac{k_2 n_2}{N}\right)}$$

El resultado $X[k_1, k_2]$ es una matriz compleja donde:
- La **magnitud** $|X[k_1,k_2]|$ indica cuánta energía tiene cada frecuencia espacial.
- La **fase** $\angle X[k_1,k_2]$ contiene información estructural de la imagen (bordes, texturas).

### 1.3 El `fftshift` — por qué centramos el espectro

Por convención, la FFT ubica la **componente DC** (frecuencia cero, que representa el brillo promedio de la imagen) en la esquina $(0,0)$ de la matriz. Aplicar `fftshift` la traslada al **centro**, facilitando la interpretación visual: las bajas frecuencias quedan en el centro y las altas en los bordes.

### 1.4 La IFFT 2D — reconstrucción de la imagen

La **Transformada Inversa Discreta de Fourier 2D (IDFT 2D)** permite regresar al dominio espacial:

$$x[n_1, n_2] = \frac{1}{MN} \sum_{k_1=0}^{M-1} \sum_{k_2=0}^{N-1} X[k_1, k_2]\, e^{\,j2\pi\!\left(\frac{k_1 n_1}{M} + \frac{k_2 n_2}{N}\right)}$$

Al modificar $X[k_1, k_2]$ antes de aplicar la IFFT (poner ciertas frecuencias en cero), obtenemos una imagen filtrada. Esto es posible gracias al **Teorema de Convolución**:

$$x[n_1,n_2] * h[n_1,n_2] \;\longleftrightarrow\; X[k_1,k_2] \cdot H[k_1,k_2]$$

Filtrar multiplicando en frecuencia es exactamente equivalente a convolucionar en el dominio espacial, pero computacionalmente mucho más eficiente.

### 1.5 Complejidad computacional: DFT vs FFT

| Algoritmo | Operaciones | Para $N = 1024$ |
|---|---|---|
| DFT directa | $\mathcal{O}(N^2)$ | $\approx 1{,}048{,}576$ |
| FFT (Cooley-Tukey) | $\mathcal{O}(N \log N)$ | $\approx 10{,}240$ |

La FFT es aproximadamente **100 veces más rápida** para $N=1024$. En 2D, la ventaja se eleva al cuadrado.

---
## Sección 2 — Carga y preparación de la imagen

Usamos una imagen estándar de prueba (`camera` de scikit-image) para garantizar que los experimentos sean **reproducibles** sin depender de un archivo externo. Más adelante, en la Sección 6, aplicamos el mismo pipeline a una imagen real.

In [ ]:
# Cargamos la imagen de prueba estándar 'camera' (512x512, escala de grises)
# Esta imagen es ampliamente usada en procesamiento de imágenes por su riqueza de detalles
img_original = data.camera().astype(np.float64)

print(f"Dimensiones de la imagen : {img_original.shape}")
print(f"Tipo de dato             : {img_original.dtype}")
print(f"Valor mínimo de píxel    : {img_original.min():.1f}")
print(f"Valor máximo de píxel    : {img_original.max():.1f}")
print(f"Total de píxeles         : {img_original.size:,}")

# Visualizamos la imagen original
plt.figure(figsize=(5, 5))
plt.imshow(img_original, cmap='gray')
plt.title("Imagen original (escala de grises)")
plt.colorbar(label='Intensidad')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Sección 3 — Análisis espectral con FFT 2D

Aplicamos la FFT 2D y visualizamos el espectro de magnitud. Para que sea visualmente interpretable usamos dos transformaciones:

1. **`fftshift`**: centra la componente DC.
2. **Escala logarítmica**: $\log(1 + |X|)$, porque la magnitud varía en varios órdenes de magnitud y sin esta escala los detalles de baja energía son invisibles.

In [ ]:
# ─── Paso 1: calcular la FFT 2D ───────────────────────────────────────────────
fft_imagen = np.fft.fft2(img_original)

# ─── Paso 2: centrar el espectro (DC al centro) ───────────────────────────────
fft_centrada = np.fft.fftshift(fft_imagen)

# ─── Paso 3: calcular magnitud en escala logarítmica ─────────────────────────
# log(1 + |X|) evita log(0) y comprime el rango dinámico para visualización
magnitud_log = np.log1p(np.abs(fft_centrada))

# ─── Visualización ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img_original, cmap='gray')
axes[0].set_title("Imagen original")
axes[0].axis('off')

im = axes[1].imshow(magnitud_log, cmap='inferno')
axes[1].set_title("Espectro de magnitud (escala log)\nDC centrado")
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], label='log(1 + |X[k₁,k₂]|)')

plt.suptitle("FFT 2D — Dominio espacial vs. dominio de frecuencias", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretación del espectro:")
print("  • Centro brillante : componente DC (brillo promedio de la imagen)")
print("  • Zona central     : bajas frecuencias (formas, estructuras generales)")
print("  • Zona periférica  : altas frecuencias (bordes, detalles finos, ruido)")

---
## Sección 4 — Diseño de filtros en el dominio de frecuencias

Definimos funciones reutilizables para construir **máscaras de filtrado**. Una máscara es una matriz binaria del mismo tamaño que el espectro: los valores `1` conservan esa frecuencia y los valores `0` la eliminan.

### Tipos de filtro implementados

| Filtro | Qué pasa | Qué elimina | Uso típico |
|---|---|---|---|
| **Pasa Bajas** | Frecuencias bajas (centro) | Frecuencias altas (bordes) | Suavizado, reducción de ruido de alta frec. |
| **Pasa Altas** | Frecuencias altas (bordes) | Frecuencias bajas (centro) | Detección de bordes, realce de detalles |
| **Notch** | Todo excepto una banda | Frecuencia puntual + entorno | Eliminación de ruido periódico |

Todos los filtros circulares se basan en la distancia euclidiana desde el centro del espectro:

$$d(k_1, k_2) = \sqrt{(k_1 - c_r)^2 + (k_2 - c_c)^2}$$

donde $(c_r, c_c)$ es la posición del centro.

In [ ]:
def crear_mascara_distancia(forma, centro=None):
    """
    Calcula la matriz de distancias euclidianas desde el centro del espectro.
    Es la base para todos los filtros circulares.
    
    Parámetros
    ----------
    forma  : tuple (filas, columnas) — tamaño del espectro
    centro : tuple (fila, columna)  — posición del centro; si None, usa el centro geométrico
    
    Retorna
    -------
    D : ndarray (filas x columnas) — distancia de cada punto al centro
    """
    filas, cols = forma
    if centro is None:
        centro = (filas // 2, cols // 2)
    
    # Creamos grillas de índices para filas y columnas
    u = np.arange(filas) - centro[0]   # distancias verticales al centro
    v = np.arange(cols)  - centro[1]   # distancias horizontales al centro
    
    # meshgrid genera todas las combinaciones (u,v) de manera vectorizada
    V, U = np.meshgrid(v, u)
    
    # Distancia euclidiana d = sqrt(u² + v²)
    D = np.sqrt(U**2 + V**2)
    return D


def filtro_pasa_bajas(forma, radio):
    """
    Máscara circular: conserva todas las frecuencias dentro del radio dado.
    
    Parámetros
    ----------
    forma : tuple — tamaño del espectro
    radio : float — radio del círculo de paso (en píxeles del espectro)
    
    Retorna
    -------
    mascara : ndarray binaria (1 dentro del radio, 0 fuera)
    """
    D = crear_mascara_distancia(forma)
    mascara = (D <= radio).astype(float)  # 1 si está dentro del radio, 0 si está fuera
    return mascara


def filtro_pasa_altas(forma, radio):
    """
    Complemento del pasa bajas: elimina las frecuencias dentro del radio.
    Realza bordes y detalles finos.
    """
    return 1.0 - filtro_pasa_bajas(forma, radio)


def filtro_notch(forma, centros_notch, radio_notch):
    """
    Elimina frecuencias puntuales específicas (útil para ruido periódico).
    Puede recibir múltiples centros para eliminar varios picos a la vez.
    
    Parámetros
    ----------
    forma         : tuple — tamaño del espectro
    centros_notch : list de tuples — posiciones (fila, col) a eliminar en el espectro centrado
    radio_notch   : float — radio de la zona a eliminar alrededor de cada centro
    """
    mascara = np.ones(forma)  # empezamos con todo en 1 (nada bloqueado)
    
    for centro in centros_notch:
        D = crear_mascara_distancia(forma, centro=centro)
        mascara[D <= radio_notch] = 0  # bloqueamos la zona alrededor del pico
    
    return mascara


def aplicar_filtro(fft_centrada, mascara):
    """
    Aplica una máscara al espectro centrado y reconstruye la imagen con IFFT.
    
    Parámetros
    ----------
    fft_centrada : ndarray complejo — espectro centrado (resultado de fftshift)
    mascara      : ndarray — máscara binaria del mismo tamaño
    
    Retorna
    -------
    img_filtrada : ndarray float — imagen reconstruida en el dominio espacial
    fft_filtrada : ndarray complejo — espectro después de aplicar la máscara
    """
    # Multiplicamos el espectro por la máscara (zeroing de frecuencias no deseadas)
    fft_filtrada = fft_centrada * mascara
    
    # Descentramos el espectro (regresamos DC a la esquina) antes de la IFFT
    fft_descentrada = np.fft.ifftshift(fft_filtrada)
    
    # Aplicamos la IFFT 2D para regresar al dominio espacial
    img_reconstruida = np.fft.ifft2(fft_descentrada)
    
    # Tomamos la parte real (los residuos imaginarios son errores numéricos ~1e-13)
    img_filtrada = np.real(img_reconstruida)
    
    return img_filtrada, fft_filtrada


print("✓ Funciones de filtrado definidas")
print("  Disponibles: filtro_pasa_bajas(), filtro_pasa_altas(), filtro_notch(), aplicar_filtro()")

---
## Sección 5 — Experimentos de filtrado

### Experimento 5.1 — Filtro Pasa Bajas: efecto del radio

El **radio** del filtro pasa bajas determina qué fracción del espectro se conserva. Un radio pequeño suaviza fuertemente la imagen (pierde detalles); un radio grande conserva más información. Comparamos tres radios para visualizar este efecto.

In [ ]:
radios = [10, 30, 80]

fig, axes = plt.subplots(2, len(radios) + 1, figsize=(16, 8))

# Columna 0: imagen original y su espectro
axes[0, 0].imshow(img_original, cmap='gray')
axes[0, 0].set_title("Original")
axes[0, 0].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(fft_centrada)), cmap='inferno')
axes[1, 0].set_title("Espectro original")
axes[1, 0].axis('off')

for idx, radio in enumerate(radios):
    # Construimos la máscara para este radio
    mascara = filtro_pasa_bajas(img_original.shape, radio)
    
    # Aplicamos el filtro y obtenemos la imagen reconstruida
    img_filtrada, fft_fil = aplicar_filtro(fft_centrada, mascara)
    
    # Calculamos qué porcentaje del espectro conservamos
    pct = mascara.mean() * 100
    
    # Fila superior: imágenes filtradas
    axes[0, idx + 1].imshow(img_filtrada, cmap='gray')
    axes[0, idx + 1].set_title(f"Pasa Bajas r={radio}\n({pct:.1f}% del espectro)")
    axes[0, idx + 1].axis('off')
    
    # Fila inferior: espectro filtrado (con máscara superpuesta)
    espectro_filtrado_vis = np.log1p(np.abs(fft_fil))
    axes[1, idx + 1].imshow(espectro_filtrado_vis, cmap='inferno')
    axes[1, idx + 1].set_title(f"Espectro filtrado r={radio}")
    axes[1, idx + 1].axis('off')

plt.suptitle("Experimento 1: Filtro Pasa Bajas con distintos radios", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observaciones:")
print("  • Radio pequeño (10) → imagen muy borrosa, pierde casi todos los detalles.")
print("  • Radio medio   (30) → suavizado moderado, se pierden texturas finas.")
print("  • Radio grande  (80) → suavizado leve, la imagen luce casi igual a la original.")

### Experimento 5.2 — Filtro Pasa Altas: detección de bordes

El filtro pasa altas elimina las frecuencias bajas (el "fondo" de la imagen) y conserva las altas, que corresponden a **cambios bruscos de intensidad**: bordes y contornos. Este es el principio detrás de muchos detectores de bordes clásicos.

In [ ]:
radios_pa = [10, 30, 60]

fig, axes = plt.subplots(2, len(radios_pa) + 1, figsize=(16, 8))

# Columna 0: original
axes[0, 0].imshow(img_original, cmap='gray')
axes[0, 0].set_title("Original")
axes[0, 0].axis('off')
axes[1, 0].imshow(np.log1p(np.abs(fft_centrada)), cmap='inferno')
axes[1, 0].set_title("Espectro original")
axes[1, 0].axis('off')

for idx, radio in enumerate(radios_pa):
    mascara = filtro_pasa_altas(img_original.shape, radio)
    img_filtrada, fft_fil = aplicar_filtro(fft_centrada, mascara)
    
    # Normalizamos para visualización (los valores pueden ser negativos tras pasa altas)
    img_vis = img_filtrada - img_filtrada.min()
    img_vis = img_vis / img_vis.max()
    
    pct = mascara.mean() * 100
    
    axes[0, idx + 1].imshow(img_vis, cmap='gray')
    axes[0, idx + 1].set_title(f"Pasa Altas r={radio}\n({pct:.1f}% del espectro)")
    axes[0, idx + 1].axis('off')
    
    axes[1, idx + 1].imshow(np.log1p(np.abs(fft_fil)), cmap='inferno')
    axes[1, idx + 1].set_title(f"Espectro filtrado r={radio}")
    axes[1, idx + 1].axis('off')

plt.suptitle("Experimento 2: Filtro Pasa Altas — Detección de bordes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observaciones:")
print("  • El filtro elimina la información de brillo global y revela solo los bordes.")
print("  • A mayor radio bloqueado, más agresiva es la detección de bordes.")
print("  • El 'halo' alrededor de los bordes es el efecto Gibbs (ringing artifact).")

### Experimento 5.3 — Filtro Notch: eliminación de ruido periódico

El ruido periódico (como el producido por la frecuencia de red eléctrica en cámaras analógicas, o patrones de impresión) genera **picos simétricos** muy pronunciados en el espectro de Fourier. El filtro Notch los elimina quirúrgicamente sin afectar el resto de la imagen.

En este experimento:
1. Añadimos artificialmente un **patrón de ruido sinusoidal** a la imagen.
2. Identificamos sus picos en el espectro.
3. Aplicamos el filtro Notch para eliminarlos.

In [ ]:
# ─── Paso 1: crear ruido periódico sinusoidal ─────────────────────────────────
filas, cols = img_original.shape

# Creamos grillas de coordenadas normalizadas
x = np.arange(cols)
y = np.arange(filas)
X, Y = np.meshgrid(x, y)

# Patrón sinusoidal: frecuencia elegida para que sus picos sean visibles en el espectro
# freq_u y freq_v son la cantidad de ciclos horizontales y verticales en la imagen
freq_u, freq_v = 30, 40
amplitud = 60   # amplitud del ruido (0-255)

ruido = amplitud * np.sin(2 * np.pi * (freq_u * X / cols + freq_v * Y / filas))

# Sumamos el ruido a la imagen y limitamos el rango a [0, 255]
img_ruidosa = np.clip(img_original + ruido, 0, 255)

# ─── Paso 2: calcular FFT de la imagen ruidosa ────────────────────────────────
fft_ruidosa   = np.fft.fft2(img_ruidosa)
fft_ruidosa_c = np.fft.fftshift(fft_ruidosa)

# ─── Visualización: original vs. ruidosa ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(img_original, cmap='gray')
axes[0].set_title("Imagen original")
axes[0].axis('off')

axes[1].imshow(img_ruidosa, cmap='gray')
axes[1].set_title(f"Imagen con ruido periódico\n(freq_u={freq_u}, freq_v={freq_v})")
axes[1].axis('off')

axes[2].imshow(np.log1p(np.abs(fft_ruidosa_c)), cmap='inferno')
axes[2].set_title("Espectro: se ven picos extra\npor el ruido periódico")
axes[2].axis('off')

plt.suptitle("Experimento 3a: Imagen con ruido periódico", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Paso 3: identificar y eliminar los picos de ruido con filtro Notch ───────
#
# Un patrón sinusoidal de frecuencias (freq_u, freq_v) produce picos en el espectro
# centrado en las posiciones:
#   (centro_fila ± freq_v, centro_col ± freq_u)
# porque el fftshift ubica la DC en (M//2, N//2).

cr, cc = filas // 2, cols // 2   # centro del espectro

# Los 4 picos simétricos del patrón sinusoidal
picos = [
    (cr + freq_v, cc + freq_u),
    (cr - freq_v, cc - freq_u),
    (cr + freq_v, cc - freq_u),
    (cr - freq_v, cc + freq_u),
]

# Construimos la máscara notch con un radio de 5 píxeles alrededor de cada pico
mascara_notch = filtro_notch(img_original.shape, centros_notch=picos, radio_notch=5)

# Aplicamos el filtro y reconstruimos
img_notch, fft_notch = aplicar_filtro(fft_ruidosa_c, mascara_notch)
img_notch = np.clip(img_notch, 0, 255)   # limitamos el rango

# ─── Visualización comparativa ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes[0, 0].imshow(img_original, cmap='gray')
axes[0, 0].set_title("Original")
axes[0, 0].axis('off')

axes[0, 1].imshow(img_ruidosa, cmap='gray')
axes[0, 1].set_title("Con ruido periódico")
axes[0, 1].axis('off')

axes[0, 2].imshow(img_notch, cmap='gray')
axes[0, 2].set_title("Recuperada con filtro Notch")
axes[0, 2].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(fft_centrada)), cmap='inferno')
axes[1, 0].set_title("Espectro original")
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log1p(np.abs(fft_ruidosa_c)), cmap='inferno')
axes[1, 1].set_title("Espectro ruidoso\n(picos extra visibles)")
axes[1, 1].axis('off')

axes[1, 2].imshow(np.log1p(np.abs(fft_notch)), cmap='inferno')
axes[1, 2].set_title("Espectro tras filtro Notch\n(picos eliminados)")
axes[1, 2].axis('off')

plt.suptitle("Experimento 3b: Filtro Notch — Eliminación de ruido periódico", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observación clave:")
print("  • Los picos del espectro (brillos puntuales) desaparecen tras el Notch.")
print("  • El patrón de líneas en la imagen ruidosa se elimina preservando la estructura.")

---
## Sección 6 — Reducción de ruido gaussiano con filtro pasa bajas

Un caso práctico muy común es la reducción de **ruido gaussiano** (el ruido aleatorio que aparece en fotografías tomadas con poca luz). El ruido gaussiano se distribuye por todo el espectro, pero tiende a concentrarse en las frecuencias altas. Un filtro pasa bajas puede reducirlo a costo de cierta nitidez.

In [ ]:
# ─── Generamos imagen con ruido gaussiano ─────────────────────────────────────
rng = np.random.default_rng(seed=42)   # semilla fija para reproducibilidad
sigma_ruido = 25   # desviación estándar del ruido (mayor sigma = más ruido)

ruido_gaussiano = rng.normal(0, sigma_ruido, img_original.shape)
img_gaussiana   = np.clip(img_original + ruido_gaussiano, 0, 255)

# ─── Aplicamos filtro pasa bajas con radio moderado ───────────────────────────
fft_gauss   = np.fft.fft2(img_gaussiana)
fft_gauss_c = np.fft.fftshift(fft_gauss)

radio_suavizado = 50
mascara_pb      = filtro_pasa_bajas(img_gaussiana.shape, radio_suavizado)
img_suavizada, _ = aplicar_filtro(fft_gauss_c, mascara_pb)
img_suavizada    = np.clip(img_suavizada, 0, 255)

# ─── Visualización ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_original, cmap='gray')
axes[0].set_title("Original (referencia)")
axes[0].axis('off')

axes[1].imshow(img_gaussiana, cmap='gray')
axes[1].set_title(f"Con ruido gaussiano\n(σ = {sigma_ruido})")
axes[1].axis('off')

axes[2].imshow(img_suavizada, cmap='gray')
axes[2].set_title(f"Filtrada (pasa bajas r={radio_suavizado})")
axes[2].axis('off')

plt.suptitle("Experimento 4: Reducción de ruido gaussiano", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Sección 7 — Evaluación cuantitativa con métricas

Para demostrar objetivamente que el filtrado mejoró la calidad de la imagen, calculamos dos métricas estándar en procesamiento de imágenes:

### Error Cuadrático Medio (MSE)

$$\text{MSE} = \frac{1}{MN} \sum_{n_1=0}^{M-1} \sum_{n_2=0}^{N-1} \left[ x[n_1,n_2] - \hat{x}[n_1,n_2] \right]^2$$

donde $x$ es la imagen de referencia (original) y $\hat{x}$ es la imagen procesada. Un MSE **menor** indica mayor fidelidad con la original.

### Peak Signal-to-Noise Ratio (PSNR)

$$\text{PSNR} = 10 \cdot \log_{10}\!\left(\frac{\text{MAX}^2}{\text{MSE}}\right) \quad [\text{dB}]$$

donde MAX = 255 para imágenes de 8 bits. Un PSNR **mayor** indica mejor calidad. Valores superiores a 30 dB se consideran aceptables; sobre 40 dB la diferencia con la original es casi imperceptible.

### Índice de Similitud Estructural (SSIM)

El SSIM mide la similitud perceptual entre dos imágenes considerando luminancia, contraste y estructura. Su valor va de 0 a 1, siendo 1 la imagen idéntica.

In [ ]:
def calcular_metricas(referencia, procesada, nombre=""):
    """
    Calcula MSE, PSNR y SSIM entre una imagen de referencia y una procesada.
    Imprime un reporte formateado.
    
    Parámetros
    ----------
    referencia : ndarray — imagen de referencia (ground truth)
    procesada  : ndarray — imagen a evaluar
    nombre     : str     — etiqueta para el reporte
    
    Retorna
    -------
    dict con {'mse', 'psnr', 'ssim'}
    """
    ref = referencia.astype(np.float64)
    proc = procesada.astype(np.float64)
    
    # MSE manual
    mse = np.mean((ref - proc) ** 2)
    
    # PSNR: indefinido si MSE = 0 (imágenes idénticas)
    if mse == 0:
        psnr = float('inf')
    else:
        psnr = 10 * np.log10((255.0 ** 2) / mse)
    
    # SSIM usando scikit-image (data_range especifica el rango de la imagen)
    ssim = metrics.structural_similarity(ref, proc, data_range=255.0)
    
    if nombre:
        print(f"  ── {nombre} ──")
    print(f"  MSE  : {mse:>10.4f}   (menor es mejor)")
    print(f"  PSNR : {psnr:>10.4f} dB (mayor es mejor)")
    print(f"  SSIM : {ssim:>10.4f}   (1 = idéntica)")
    print()
    
    return {'mse': mse, 'psnr': psnr, 'ssim': ssim}


print("=" * 50)
print("REPORTE DE MÉTRICAS — Experimento Ruido Gaussiano")
print("=" * 50)

print()
m_ruidosa   = calcular_metricas(img_original, img_gaussiana,  "Imagen ruidosa vs. original")
m_suavizada = calcular_metricas(img_original, img_suavizada,  "Imagen filtrada vs. original")

# ─── Resumen comparativo ──────────────────────────────────────────────────────
print("=" * 50)
print("RESUMEN COMPARATIVO")
print("=" * 50)
print(f"{'Métrica':<10} {'Ruidosa':>12} {'Filtrada':>12} {'Mejora':>10}")
print("-" * 46)
print(f"{'MSE':<10} {m_ruidosa['mse']:>12.4f} {m_suavizada['mse']:>12.4f} {(1 - m_suavizada['mse']/m_ruidosa['mse'])*100:>9.1f}%")
print(f"{'PSNR (dB)':<10} {m_ruidosa['psnr']:>12.4f} {m_suavizada['psnr']:>12.4f} {m_suavizada['psnr'] - m_ruidosa['psnr']:>+9.4f}")
print(f"{'SSIM':<10} {m_ruidosa['ssim']:>12.4f} {m_suavizada['ssim']:>12.4f} {m_suavizada['ssim'] - m_ruidosa['ssim']:>+9.4f}")

---
## Sección 8 — Comparación visual completa de todos los experimentos

In [ ]:
# Recalculamos versiones limpias para el panel final
mascara_pb_30, _ = aplicar_filtro(fft_centrada, filtro_pasa_bajas(img_original.shape, 30))
mascara_pa_30, _ = aplicar_filtro(fft_centrada, filtro_pasa_altas(img_original.shape, 30))
img_pa_vis = mascara_pa_30 - mascara_pa_30.min()
img_pa_vis = img_pa_vis / img_pa_vis.max() * 255

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

experimentos = [
    (img_original,        "Original",                                       np.log1p(np.abs(fft_centrada))),
    (mascara_pb_30,       "Pasa Bajas (r=30)\nSuavizado",                   np.log1p(np.abs(aplicar_filtro(fft_centrada, filtro_pasa_bajas(img_original.shape, 30))[1]))),
    (img_pa_vis,          "Pasa Altas (r=30)\nDetección de bordes",         np.log1p(np.abs(aplicar_filtro(fft_centrada, filtro_pasa_altas(img_original.shape, 30))[1]))),
    (img_notch,           "Notch\nEliminación de ruido periódico",          np.log1p(np.abs(fft_notch))),
]

for col, (img, titulo, espectro) in enumerate(experimentos):
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(titulo)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(espectro, cmap='inferno')
    axes[1, col].set_title("Espectro")
    axes[1, col].axis('off')

plt.suptitle("Panel comparativo — Todos los filtros implementados", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Sección 9 — Benchmark: DFT directa O(N²) vs FFT O(N log N)

La FFT no es una transformada diferente a la DFT: es el **mismo resultado**, calculado de forma más eficiente. Implementamos la DFT 2D desde cero con cuatro bucles anidados para comparar tiempos de ejecución.

> **Nota**: la DFT manual se ejecuta sobre una imagen muy pequeña (32×32 píxeles) porque su complejidad $\mathcal{O}(N^4)$ en 2D haría inviable el cálculo para tamaños normales.

In [ ]:
import time

def dft_2d_manual(imagen):
    """
    Implementación directa de la DFT 2D sin optimizaciones.
    Complejidad: O(M² × N²) — extremadamente lento para imágenes grandes.
    
    Solo con fines demostrativos para mostrar el costo computacional
    que la FFT reduce drásticamente.
    """
    M, N = imagen.shape
    X = np.zeros((M, N), dtype=complex)   # arreglo de salida complejo
    
    for k1 in range(M):         # índice de frecuencia vertical
        for k2 in range(N):     # índice de frecuencia horizontal
            suma = 0 + 0j
            for n1 in range(M): # índice espacial vertical
                for n2 in range(N): # índice espacial horizontal
                    # Kernel de la DFT: e^{-j2π(k1*n1/M + k2*n2/N)}
                    exponente = -1j * 2 * np.pi * (k1 * n1 / M + k2 * n2 / N)
                    suma += imagen[n1, n2] * np.exp(exponente)
            X[k1, k2] = suma
    return X


# Usamos una imagen muy pequeña para que la DFT manual sea computable
tamano_benchmark = 32
img_pequena = img_original[:tamano_benchmark, :tamano_benchmark]

print(f"Tamaño de prueba: {tamano_benchmark}×{tamano_benchmark} = {tamano_benchmark**2} píxeles")
print(f"Operaciones DFT manual: ~{tamano_benchmark**4:,} (N⁴)")
print(f"Operaciones FFT 2D:     ~{int(tamano_benchmark**2 * np.log2(tamano_benchmark**2)):,} (N² log N²)")
print()

# ─── Benchmark DFT manual ─────────────────────────────────────────────────────
print("Ejecutando DFT manual...", end=" ", flush=True)
t0 = time.perf_counter()
resultado_dft = dft_2d_manual(img_pequena)
t_dft = time.perf_counter() - t0
print(f"✓  Tiempo: {t_dft:.4f} s")

# ─── Benchmark FFT de NumPy ───────────────────────────────────────────────────
print("Ejecutando FFT NumPy...", end=" ", flush=True)
t0 = time.perf_counter()
resultado_fft = np.fft.fft2(img_pequena)
t_fft = time.perf_counter() - t0
print(f"✓  Tiempo: {t_fft:.6f} s")

# ─── Verificación de correctitud ──────────────────────────────────────────────
error_maximo = np.max(np.abs(resultado_dft - resultado_fft))
factor       = t_dft / t_fft

print()
print("=" * 45)
print(f"Error máximo entre DFT y FFT: {error_maximo:.2e}")
print(f"  → Ambas producen el mismo resultado numérico.")
print()
print(f"Factor de aceleración: {factor:.0f}× más rápida la FFT")
print("=" * 45)

---
## Sección 10 — Conclusiones

### Lo que demostramos en este notebook

1. **La FFT 2D permite analizar imágenes en el dominio de frecuencias**, donde el centro del espectro representa estructuras globales y la periferia representa detalles finos.

2. **El filtrado espectral es poderoso y flexible**: con solo multiplicar el espectro por una máscara, podemos suavizar, realzar bordes, o eliminar ruido periódico, y reconstruir la imagen con la IFFT.

3. **Las métricas confirman cuantitativamente** que el filtrado mejora la calidad de la imagen respecto de la versión ruidosa, y permiten elegir el radio óptimo.

4. **La FFT es exactamente la DFT**, solo que calculada de forma eficiente. El error entre ambos métodos es del orden de $10^{-10}$ (error numérico de punto flotante), y la FFT puede ser cientos de veces más rápida.

### Limitaciones del enfoque

- Los filtros abruptos (máscara binaria) generan el **efecto de ringing** (artefactos de oscilación alrededor de bordes), relacionado con el fenómeno de Gibbs.
- El filtrado funciona mejor con ruido **estacionario**; el ruido que varía en el espacio requiere técnicas más avanzadas.
- La máscara circular asume isotropía frecuencial; patrones de ruido direccionales requieren máscaras más elaboradas.

### Referencias

- Tan, X. "Fast Fourier Transform". University of Connecticut, 2020.
- Kulkarni, S. R. "Frequency Domain and Fourier Transforms". Princeton University.
- Python Numerical Methods at Berkeley. *Discrete Fourier Transform*.
- NumPy Documentation: `numpy.fft` module.
- scikit-image Documentation: `skimage.metrics` module.